Link to the model: https://drive.google.com/file/d/1z3B6FT1yDqhhbTsw__Wmy5q9Omt94ZBQ/view?usp=sharing

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    set_seed
)
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

In [2]:
def parse_conll(filepath):
    """
    Parse a CoNLL-formatted file into a list of sentences.
    Args:
        filepath (str): Path to the CoNLL file.

    Returns:
        list[list[list[str]]]: A list of sentences. Each sentence is a list
        of token rows, and each row is a list of column values.
    """
    sentences = []
    current_sentence = []
    
    with open(filepath, 'r', encoding='utf-8') as f: 
        for line in f:
            line = line.strip() #remove whitespace
            
            if line.startswith('#'):
                continue  # skip comment lines
            
            if line == '':  #Empty line shows end of sentence
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
                continue
            
            cols = line.split('\t')
            current_sentence.append(cols)
    
    if current_sentence:
        sentences.append(current_sentence)
    
    return sentences

In [3]:
def clean_label(label):
    """
    Maps '_' (no semantic role) and predicates itself to 'O' (outside).
    Args:
        label (str): Original label from the dataset.

    Returns:
        str: Normalized label.
    """
    if label == '_' or label == 'V':
        return 'O'
    return label


def preprocess(sentences):
    """
    Convert parsed CoNLL sentences into SRL instances (one per predicate)
    Args:
        sentences (list[list[list[str]]]): Output from `parse_conll`.

    Returns:
        list[dict]: A list of SRL instances, each containing:
            {'tokens': list[str],      
                'predicate_idx': int,     
                'labels': list[str],       
                'sent': original sentence}  
    """
    instances = []

    for sent in sentences:
        #Find indices of predicates
        predicate_indices = [
            i for i, tok in enumerate(sent)
            if len(tok) > 10 and tok[10] != '_'
        ]

        if not predicate_indices:   # skip sents without predicates bc SRL is predicate-dependent
            continue
        # Create one instance per predicate
        for pred_num, pred_idx in enumerate(predicate_indices):
            tokens = [tok[1] for tok in sent] #Extract tokens (column 1)
            labels = []

            for tok in sent:
                column_index = 11 + pred_num  #argument column for this predicate

                # Check if this token even has this column
                if len(tok) <= column_index:
                    label = 'O'

                #if the column value is '_' or 'V', map to 'O'
                elif tok[column_index] == '_' or tok[column_index] == 'V':
                    label = 'O'

                # Otherwise use the actual SRL label
                else:
                    label = tok[column_index]

                labels.append(label)

            instances.append({
                'tokens': tokens,
                'predicate_idx': pred_idx,
                'labels': labels,
                'sent': sent
            })

    return instances

In [4]:
train_sentences = parse_conll('data/en_ewt-up-train.conllu')
test_sentences = parse_conll('data/en_ewt-up-test.conllu')
train_instances = preprocess(train_sentences)
test_instances  = preprocess(test_sentences)

In [5]:
def count_tokens(instances):
    return sum(len(i['tokens']) for i in instances)

def count_sentences(instances):
    return len(instances)

print("train")
print(f"Sentences before preprocessing : {len(train_sentences)}")
print(f"Sentences after preprocessing  : {len(train_instances)}")
print(f"Tokens before preprocessing    : {sum(len(s) for s in train_sentences)}")
print(f"Tokens after preprocessing     : {count_tokens(train_instances)}")

print("\ntest")
print(f"Sentences before preprocessing : {len(test_sentences)}")
print(f"Sentences after preprocessing  : {len(test_instances)}")
print(f"Tokens before preprocessing    : {sum(len(s) for s in test_sentences)}")
print(f"Tokens after preprocessing     : {count_tokens(test_instances)}")

train
Sentences before preprocessing : 12543
Sentences after preprocessing  : 40482
Tokens before preprocessing    : 204609
Tokens after preprocessing     : 1028264

test
Sentences before preprocessing : 2077
Sentences after preprocessing  : 4799
Tokens before preprocessing    : 25097
Tokens after preprocessing     : 101152


In [6]:
train_label_set = set(label for inst in train_instances for label in inst['labels'])
test_label_set  = set(label for inst in test_instances  for label in inst['labels'])

unseen = test_label_set - train_label_set
print("In test but not train:", unseen)

for label in unseen:
    count = sum(inst['labels'].count(label) for inst in test_instances)
    print(f"  '{label}' appears {count} times in test set")

In test but not train: {'R-ARGM-ADJ'}
  'R-ARGM-ADJ' appears 1 times in test set


In [7]:
#label vocabulary
all_labels = set()
for instance in train_instances + test_instances:   #added test to avoid alignment crash (bc there is one label in test as shown above)
    all_labels.update(instance['labels'])

label_list = sorted(all_labels)
label2id   = {label: i for i, label in enumerate(label_list)}
id2label   = {i: label for i, label in enumerate(label_list)}

print(f"Number of labels: {len(label_list)}")
print(label_list)

Number of labels: 60
['ARG0', 'ARG1', 'ARG1-DSP', 'ARG2', 'ARG3', 'ARG4', 'ARG5', 'ARGA', 'ARGM-ADJ', 'ARGM-ADV', 'ARGM-CAU', 'ARGM-COM', 'ARGM-CXN', 'ARGM-DIR', 'ARGM-DIS', 'ARGM-EXT', 'ARGM-GOL', 'ARGM-LOC', 'ARGM-LVB', 'ARGM-MNR', 'ARGM-MOD', 'ARGM-NEG', 'ARGM-PRD', 'ARGM-PRP', 'ARGM-PRR', 'ARGM-REC', 'ARGM-TMP', 'C-ARG0', 'C-ARG1', 'C-ARG1-DSP', 'C-ARG2', 'C-ARG3', 'C-ARG4', 'C-ARGM-ADV', 'C-ARGM-COM', 'C-ARGM-CXN', 'C-ARGM-DIR', 'C-ARGM-EXT', 'C-ARGM-GOL', 'C-ARGM-LOC', 'C-ARGM-MNR', 'C-ARGM-PRP', 'C-ARGM-PRR', 'C-ARGM-TMP', 'C-V', 'O', 'R-ARG0', 'R-ARG1', 'R-ARG2', 'R-ARG3', 'R-ARG4', 'R-ARGM-ADJ', 'R-ARGM-ADV', 'R-ARGM-CAU', 'R-ARGM-COM', 'R-ARGM-DIR', 'R-ARGM-GOL', 'R-ARGM-LOC', 'R-ARGM-MNR', 'R-ARGM-TMP']


## Model Selection
I use `distilbert-base-uncased`, which is a distilled version of BERT that retains most of BERT's performance while being smaller and faster. It is suitable for token classification tasks like SRL, and makes fine-tuning faster withouth much difference in performance.

In [8]:
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Number of labels: {len(label_list)}")

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizer: DistilBertTokenizerFast
Vocabulary size: 30522
Number of labels: 60


In [14]:
#add predicate markers as special tokens 
new_tokens = ['[PRED]', '[/PRED]']
tokenizer.add_special_tokens({'additional_special_tokens': new_tokens})
model.resize_token_embeddings(len(tokenizer))
print(f"Added special tokens: {new_tokens}")

Added special tokens: ['[PRED]', '[/PRED]']


## Input Preprocessing: Predicate Marking
The same sentence should processed once per predicate, and the model needs to know which predicate's arguments it is labeling, as SRL labels depend on the specific predicate.
Following Shi & Lin (2019), I made the predicate explicit in the input by wrapping the predicate token with [PRED] and [/PRED] markers. 
For example:
    Original: ['forces', 'killed', 'him']
    Marked:   ['forces', '[PRED]killed[/PRED]', 'him']

This approach makes predicate information explicit in the token sequence so the model can adjust its contextual representations on which verb is being labeled.

In [15]:
def mark_predicate(tokens, predicate_idx):  #making the predicate explicit in the input
    """
    Marks the predicate token by wrapping it with [PRED] markers.
    """
    marked = tokens.copy()
    marked[predicate_idx] = f"[PRED]{tokens[predicate_idx]}[/PRED]"
    return marked


#example
instance = train_instances[0]
tokens   = instance['tokens']
pred_idx = instance['predicate_idx']
labels   = instance['labels']

marked = mark_predicate(tokens, pred_idx)
print("Original tokens:", tokens)
print("Marked tokens:  ", marked)
print("Labels:         ", labels)

Original tokens: ['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
Marked tokens:   ['Al', '-', 'Zaman', ':', 'American', 'forces', '[PRED]killed[/PRED]', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
Labels:          ['O', 'O', 'O', 'O', 'O', 'ARG0', 'O', 'ARG1', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'ARGM-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [16]:
# show human-readable and machine-readable tokenization
example_tokens = mark_predicate(train_instances[0]['tokens'], train_instances[0]['predicate_idx'])
example_labels = train_instances[0]['labels']

print("Original tokens (human-readable):", example_tokens)
print("Original labels (human-readable):", example_labels)

tokenized = tokenizer(example_tokens, is_split_into_words=True)
print("\nSubword tokens:", tokenizer.convert_ids_to_tokens(tokenized['input_ids']))
print("Subword tokens (machine-readable):", tokenized['input_ids'])
print("Word IDs mapping:", tokenized.word_ids())

Original tokens (human-readable): ['Al', '-', 'Zaman', ':', 'American', 'forces', '[PRED]killed[/PRED]', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
Original labels (human-readable): ['O', 'O', 'O', 'O', 'O', 'ARG0', 'O', 'ARG1', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'ARGM-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Subword tokens: ['[CLS]', 'al', '-', 'za', '##man', ':', 'american', 'forces', '[PRED]', 'killed', '[/PRED]', 'sha', '##ikh', 'abdullah', 'al', '-', 'an', '##i', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'q', '##ai', '##m', ',', 'near', 'the', 'syrian', 'border', '.', '[SEP]']
Subword tokens (machine-readable): [101, 2632, 1011, 23564, 2386, 1024, 2137, 2749, 30522, 2730, 30523, 21146, 28209, 14093, 2632, 1011, 2019, 2072, 1010, 1996, 14512, 2012, 1996, 8806, 1999, 1996, 2237, 1997, 1053, 4886, 22

In [17]:
def create_dataset_dict(instances):
    """
    Converts instances into HuggingFace Dataset format, and marks the predicate before storing.
    """
    marked_tokens = []
    srl_labels = []
    
    for inst in instances:
        tokens = inst['tokens']
        predicate_idx = inst['predicate_idx']
        labels = inst['labels']
        
        #Mark predicate
        marked = mark_predicate(tokens, predicate_idx)
        
        marked_tokens.append(marked)
        srl_labels.append(labels)
    
    #Create dataset dictionary
    dataset = Dataset.from_dict({
        'tokens': marked_tokens,
        'srl_labels': srl_labels
    })
    
    return dataset


train_dataset = create_dataset_dict(train_instances)
test_dataset  = create_dataset_dict(test_instances)

datasets = DatasetDict({
    'train': train_dataset,
    'test':  test_dataset
})

print(datasets)
print("\nExample from training set:")
print(datasets['train'][0])

DatasetDict({
    train: Dataset({
        features: ['tokens', 'srl_labels'],
        num_rows: 40482
    })
    test: Dataset({
        features: ['tokens', 'srl_labels'],
        num_rows: 4799
    })
})

Example from training set:
{'tokens': ['Al', '-', 'Zaman', ':', 'American', 'forces', '[PRED]killed[/PRED]', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.'], 'srl_labels': ['O', 'O', 'O', 'O', 'O', 'ARG0', 'O', 'ARG1', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'ARGM-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


In [18]:
label_all_tokens = True  #label all subword tokens, not only subword

def tokenize_and_align_labels(dataset):
    """
    Tokenizes a dataset and aligns SRL labels with WordPiece subword tokens
    Args: 
        dataset (dict)
    Returns: 
        dict: A dictionary with tokenized inputs and aligned label ids
    
    """
    tokenized_inputs = tokenizer(
        dataset['tokens'],
        truncation=True,
        is_split_into_words=True #input is already tokenized
    )
    
    labels = []
    for i, label in enumerate(dataset['srl_labels']):
        # word_ids maps each subword to its original word index
        word_ids          = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids         = []
        # Align labels to subword tokens
        for word_idx in word_ids:
            if word_idx is None:
                # special token: ignore
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # first subword: use real label
                label_ids.append(label2id[label[word_idx]])
            else:
                # continuation subword
                label_ids.append(label2id[label[word_idx]] if label_all_tokens else -100)
            
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    # Add aligned labels to the tokenized input
    tokenized_inputs['labels'] = labels
    return tokenized_inputs


tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)
print(tokenized_datasets)

Map:   0%|          | 0/40482 [00:00<?, ? examples/s]

Map:   0%|          | 0/4799 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'srl_labels', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 40482
    })
    test: Dataset({
        features: ['tokens', 'srl_labels', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 4799
    })
})


In [20]:
#example of data fed into model (input_ids and attention_mask are the only inputs to the model, labels passed separately to calculate loss)
example_batch    = datasets['train'][:2]
tokenized_example = tokenize_and_align_labels(example_batch)

for ex_idx in range(2):
    print(f"Original tokens: {example_batch['tokens'][ex_idx]}")
    print(f"Original labels: {example_batch['srl_labels'][ex_idx]}")
    
    input_ids  = tokenized_example['input_ids'][ex_idx]
    lab_ids    = tokenized_example['labels'][ex_idx]
    subwords   = tokenizer.convert_ids_to_tokens(input_ids)
    label_names = [id2label[l] if l != -100 else "-" for l in lab_ids]
    
    for token, label in zip(subwords, label_names):
        print(f"  {token:<20} {label}")
    print()

Original tokens: ['Al', '-', 'Zaman', ':', 'American', 'forces', '[PRED]killed[/PRED]', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
Original labels: ['O', 'O', 'O', 'O', 'O', 'ARG0', 'O', 'ARG1', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'ARGM-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
  [CLS]                -
  al                   O
  -                    O
  za                   O
  ##man                O
  :                    O
  american             O
  forces               ARG0
  [PRED]               O
  killed               O
  [/PRED]              O
  sha                  ARG1
  ##ikh                ARG1
  abdullah             O
  al                   O
  -                    O
  an                   O
  ##i                  O
  ,                    O
  the                  O
  preacher             O
  at                   O
  the     

In [21]:
# check no label IDs appear in input_ids
label_values = set(range(len(label_list)))

def check_input_ids_for_labels(dataset):
    for split_name, split_data in dataset.items():
        for i, input_ids in enumerate(split_data['input_ids']):
            if set(input_ids) & label_values:
                print(f"Found label in input_ids at {split_name}[{i}]")
                return
    print("No labels found in input_ids.")

check_input_ids_for_labels(tokenized_datasets)

No labels found in input_ids.


In [22]:
SEED = 42
set_seed(SEED)

batch_size = 16
task       = "srl"
model_name = model_checkpoint.split("/")[-1]

args = TrainingArguments(
    f"{model_name}-finetuned-{task}",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=SEED,
    report_to=None,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    data_collator=data_collator,
    processing_class=tokenizer, 
)

trainer.train()

C:\Users\Rey\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,0.107800,0.116663
2,0.078500,0.098185
3,0.065100,0.097897


C:\Users\Rey\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\Rey\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\Rey\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=7593, training_loss=0.11079341068767706, metrics={'train_runtime': 17733.0312, 'train_samples_per_second': 6.849, 'train_steps_per_second': 0.428, 'total_flos': 2187548500532544.0, 'train_loss': 0.11079341068767706, 'epoch': 3.0})

## Post-processing: Subword to Token Level
BERT uses WordPiece subword tokenization, and produces predictions at the subword level. Since we need token-level predictions, we need to convert subword predictions back to one prediction per original token.
I use majority voting. For each original token, all subword predictions are collected and the most frequent label is selected. In case of a tie, the prediction from the first subword is used. This is consistent with the choice made during training (label_all_tokens=True), where all subwords of a token received the same label.

Example:
    Original token: 'Antske'
    Subwords:       ['Ant', '##ske']
    Subword preds:  [ARG1, ARG1]
    Token pred:     ARG1  (majority vote)

The token count after alignment must match the number of preprocessed tokens, which I check before evaluation.

In [23]:
def align_predictions_with_tokens(predictions, original_tokens_list, loaded_tokenizer=None):
    """
    Converts subword-level predictions back to token-level predictions.
    Uses majority voting among subword predictions, and first subword prediction in case of a tie.
    """
    tok = loaded_tokenizer if loaded_tokenizer is not None else tokenizer
    token_predictions = []

    for pred_seq, orig_tokens in zip(predictions, original_tokens_list):
        # Tokenize to get word_ids mapping
        tokenized = tok(orig_tokens, is_split_into_words=True, truncation=True)
        word_ids  = tokenized.word_ids()

        # Store votes and first label for each original token
        votes        = {}
        first_labels = {}
        
        for i, word_idx in enumerate(word_ids):
            # Skip special tokens
            if word_idx is None:
                continue
            # Get label prediction for current subword
            label_id = int(pred_seq[i])
            # Group predictions for subwords with same word_idx
            if word_idx not in votes:
                votes[word_idx]        = []
                first_labels[word_idx] = label_id
            votes[word_idx].append(label_id)
            
        #majority vote for each token
        sentence_preds = []
        for word_idx in sorted(votes.keys()):
            word_votes = votes[word_idx]
            counts     = Counter(word_votes)
            max_freq   = max(counts.values()) #max frequency
            candidates = [label for label, count in counts.items() if count == max_freq] #Collect all candidates with max frequency
            
            #use first subword prediction in case of a tie
            final_label = first_labels[word_idx] if len(candidates) > 1 else candidates[0]
            sentence_preds.append(id2label[final_label])  #convert int to SRL label string
        
        token_predictions.append(sentence_preds)
    
    return token_predictions

In [24]:
# get predictions on test set
test_output  = trainer.predict(tokenized_datasets['test'])
predictions  = np.argmax(test_output.predictions, axis=2)

#post-processing example
print("=== Post-processing example ===")
example_inst   = test_instances[0]
example_tokens = mark_predicate(example_inst['tokens'], example_inst['predicate_idx'])
tokenized_ex   = tokenizer(example_tokens, is_split_into_words=True)
subwords       = tokenizer.convert_ids_to_tokens(tokenized_ex['input_ids'])
word_ids       = tokenized_ex.word_ids()

print("Original tokens:", example_tokens)
print("Subwords:       ", subwords)
print("Word IDs:       ", word_ids)
print("Subword preds:  ", predictions[0].tolist())

# align
all_marked_tokens    = [mark_predicate(inst['tokens'], inst['predicate_idx']) for inst in test_instances]
test_token_preds     = align_predictions_with_tokens(predictions, all_marked_tokens)
print("Token preds:    ", test_token_preds[0])
print("Gold labels:    ", test_instances[0]['labels'])

=== Post-processing example ===
Original tokens: ['What', 'if', 'Google', '[PRED]Morphed[/PRED]', 'Into', 'GoogleOS', '?']
Subwords:        ['[CLS]', 'what', 'if', 'google', '[PRED]', 'mor', '##ph', '##ed', '[/PRED]', 'into', 'google', '##os', '?', '[SEP]']
Word IDs:        [None, 0, 1, 2, 3, 3, 3, 3, 3, 4, 5, 5, 6, None]
Subword preds:   [45, 45, 45, 1, 45, 45, 45, 45, 45, 45, 3, 3, 45, 3, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 45, 3, 45, 45, 45, 45, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Token preds:     ['O', 'O', 'ARG1', 'O', 'O', 'ARG2', 'O']
Gold labels:     ['O', 'O', 'ARG1', 'O', 'O', 'ARG2', 'O']


In [25]:
total_predicted = sum(len(s) for s in test_token_preds)
total_gold      = sum(len(inst['labels']) for inst in test_instances)

print(f"Total predicted tokens : {total_predicted}")
print(f"Total gold tokens      : {total_gold}")

Total predicted tokens : 101152
Total gold tokens      : 101152


In [26]:
# flatten to token level
gold_flat = [label for inst in test_instances for label in inst['labels']]
pred_flat = [label for sent in test_token_preds for label in sent]

print("Classification Report")
print(classification_report(gold_flat, pred_flat, digits=3, zero_division=0))

labels_sorted = sorted(set(gold_flat))
cm     = confusion_matrix(gold_flat, pred_flat, labels=labels_sorted)
cm_df  = pd.DataFrame(cm, index=labels_sorted, columns=labels_sorted)
print("Confusion Matrix")
print(cm_df)

Classification Report
              precision    recall  f1-score   support

        ARG0      0.856     0.873     0.864      1733
        ARG1      0.858     0.878     0.868      3241
    ARG1-DSP      0.000     0.000     0.000         4
        ARG2      0.772     0.771     0.772      1129
        ARG3      0.400     0.027     0.051        74
        ARG4      0.577     0.536     0.556        56
        ARG5      0.000     0.000     0.000         1
        ARGA      0.000     0.000     0.000         2
    ARGM-ADJ      0.698     0.750     0.723       228
    ARGM-ADV      0.744     0.591     0.658       496
    ARGM-CAU      0.517     0.652     0.577        46
    ARGM-COM      0.000     0.000     0.000        13
    ARGM-CXN      1.000     0.167     0.286        12
    ARGM-DIR      0.447     0.362     0.400        47
    ARGM-DIS      0.767     0.742     0.754       182
    ARGM-EXT      0.800     0.762     0.780       105
    ARGM-GOL      1.000     0.042     0.080        24
    A

## Results Discussion
The model has 97.4% overall accuracy with a weighted average F1 of 0.974, affected by the majority label O. Main argument roles have strong performance: ARG0 (F1=0.869), ARG1 (F1=0.868), and ARG2 (F1=0.770). They all show a significant improvement over Logistic Regression, as BERT's contextual embeddings capture long-range dependencies. ARGM-MOD (F1=0.967) and ARGM-NEG (F1=0.959) are the strongest adjunct roles, likely because modal auxiliaries and negation markers have distinct POS tags and dependency relations. ARGM-TMP (F1=0.830) and ARGM-DIS (F1=0.773) also improved over A1, likely due BERT's ability to capture contextual patterns.

Rare labels with very few examples still score F1=0. ARG5 (1 instance), ARGA (2), C-ARG0 (3), and R-ARGMs, as the model never sees enough training examples to learn these patterns. The confusion matrix shows that most errors involve confusion with majority label O, or confusion between structurally similar roles such as ARG1 and ARG2. The low macro average F1 of 0.417 is affected by the zero scores of rare labels.

The results show that fine-tuning BERT outperforms the previous feature-based Logistic Regression for SRL, especially for frequent and medium-frequency roles and not taking into account data sparsity issues. The predicate marking seems ti have successfully adjusted the model on the specific predicate being labeled, which is required for SRL.

In [27]:
output_file = 'predictions_a2.tsv'

with open(output_file, 'w', encoding='utf-8') as f:
    f.write("token\tgold\tpredicted\n")
    for inst, preds in zip(test_instances, test_token_preds):
        for token, gold, pred in zip(inst['tokens'], inst['labels'], preds):
            f.write(f"{token}\t{gold}\t{pred}\n")

print(f"Predictions saved to: {output_file}")

Predictions saved to: predictions_a2.tsv


In [28]:
save_directory = f'./srl_model_seed_{SEED}'
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print("Model saved.")

Model saved.


In [29]:
def predict_srl(sentence_tokens, predicate_mask, model_path=f'./srl_model_seed_{SEED}'):
    """
    Performs SRL on a new sentence given the predicate position.
    
    Args:
        sentence_tokens: list of strings e.g. ['Pia', 'asked', 'Luis', '.']
        predicate_mask:  list of 0/1    e.g. [0, 1, 0, 0]  (1 = predicate)
        model_path:      path to saved model
    
    Returns:
        list of predicted SRL labels, one per token
    """
    predicate_idx = predicate_mask.index(1)
    
    # mark predicate 
    marked_tokens = mark_predicate(sentence_tokens, predicate_idx)
    
    # load model and tokenizer
    loaded_model     = AutoModelForTokenClassification.from_pretrained(model_path)
    loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # tokenize
    tokenized = loaded_tokenizer(
        marked_tokens,
        is_split_into_words=True,
        truncation=True,
        return_tensors="pt"
    )
    
    # predict
    loaded_trainer = Trainer(model=loaded_model)
    standalone_ds  = Dataset.from_list([{k: v[0].tolist() for k, v in tokenized.items()}])
    preds, _, _    = loaded_trainer.predict(standalone_ds)
    preds          = np.argmax(preds, axis=2)
    
    # align subwords back to tokens
    aligned = align_predictions_with_tokens(preds, [marked_tokens], loaded_tokenizer)
    return aligned[0]


# example with two predicates 
sentence = ['Pia', 'asked', 'Luis', 'to', 'write', 'this', 'sentence', '.']

print("Predicate: 'asked'")
labels = predict_srl(sentence, [0,1,0,0,0,0,0,0])
for tok, lab in zip(sentence, labels):
    print(f"  {tok:<12} {lab}")

print("\nPredicate: 'write'")
labels = predict_srl(sentence, [0,0,0,0,1,0,0,0])
for tok, lab in zip(sentence, labels):
    print(f"  {tok:<12} {lab}")

Predicate: 'asked'


C:\Users\Rey\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


  Pia          ARG0
  asked        O
  Luis         ARG2
  to           O
  write        ARG1
  this         O
  sentence     O
  .            O

Predicate: 'write'


  Pia          O
  asked        O
  Luis         ARG0
  to           O
  write        O
  this         O
  sentence     ARG1
  .            O


In [30]:
import os
print(os.getcwd())  # shows where jupyter thinks it is

C:\Users\Rey\Desktop\NLP\ass2
